# Evaluación del Modelo IA: benchmarking, sesgo algorítmico y diagnóstico de ajuste

Este notebook complementa el documento académico de evaluación. Reproduce las tablas y figuras usadas para comparar el modelo principal del proyecto con modelos base, auditar desempeño por grupos y construir curvas de aprendizaje para diagnosticar overfitting/underfitting.

**Modelo evaluado:** clasificador binario `boost_candidate` basado en Regresión Logística con TF-IDF, variables textuales, duración y categoría.

**Salida objetivo:** estimar si un video tiene potencial relativo para ser considerado candidato a impulso publicitario, sin interpretar la predicción como ROI causal.

In [ ]:
from pathlib import Path
import sys, json

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

OUT = ROOT / "outputs" / "evaluacion_modelo_ia"
OUT.mkdir(parents=True, exist_ok=True)
print(ROOT)
print(OUT)

## 1. Ejecución reproducible

El script `scripts/evaluate_model_ia_phase.py` genera automáticamente:

- `benchmark_baselines_modelo.csv`
- `fairness_por_grupos.csv`
- `fairness_resumen.csv`
- `learning_curves_logistic_regression.csv`
- figuras de curvas de aprendizaje en PNG
- `evaluacion_resumen.json`

In [ ]:
# Ejecutar esta celda para regenerar todos los resultados
# Puede tardar algunos minutos según la máquina.
import subprocess, os

env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT)
result = subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "evaluate_model_ia_phase.py")],
    cwd=str(ROOT), env=env, capture_output=True, text=True
)
print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("La evaluación falló")

## 2. Benchmarking comparativo

Se comparan tres referencias simples con el modelo propuesto: un clasificador mayoritario, un clasificador aleatorio estratificado y una regla simple basada en longitud textual. Esta sección responde si el modelo justifica su complejidad frente a un baseline trivial.

In [ ]:
import pandas as pd
benchmark = pd.read_csv(OUT / "benchmark_baselines_modelo.csv")
benchmark

In [ ]:
benchmark[["modelo", "accuracy", "precision", "recall", "f1", "roc_auc"]].sort_values("f1", ascending=False)

## 3. Análisis de sesgo algorítmico

Se evalúan variables proxy relevantes para el caso de estudio: país, categoría del video, duración y origen del conjunto de datos. Como el dataset no contiene atributos demográficos de usuarios, el análisis se interpreta como auditoría de impacto diferencial por grupos de contenido y mercado, no como certificación de equidad individual.

In [ ]:
fairness_resumen = pd.read_csv(OUT / "fairness_resumen.csv")
fairness_resumen

In [ ]:
fairness = pd.read_csv(OUT / "fairness_por_grupos.csv")
# Mostrar los grupos con mayor soporte por variable
fairness.sort_values(["variable_grupo", "n"], ascending=[True, False]).groupby("variable_grupo").head(8)

## 4. Diagnóstico de overfitting/underfitting

Las curvas de aprendizaje comparan desempeño en entrenamiento y validación. Se usa una validación fija y submuestras estratificadas de entrenamiento para observar si la brecha se reduce al aumentar los datos.

In [ ]:
learning = pd.read_csv(OUT / "learning_curves_logistic_regression.csv")
learning

### Curva de aprendizaje - F1

![Curva F1](../outputs/evaluacion_modelo_ia/curva_aprendizaje_f1.png)

### Curva de aprendizaje - Log loss

![Curva Log Loss](../outputs/evaluacion_modelo_ia/curva_aprendizaje_logloss.png)

### Curva de aprendizaje - ROC-AUC

![Curva ROC-AUC](../outputs/evaluacion_modelo_ia/curva_aprendizaje_auc.png)

In [ ]:
with open(OUT / "evaluacion_resumen.json", encoding="utf-8") as f:
    resumen = json.load(f)
resumen

## 5. Lectura técnica breve

- El modelo propuesto supera a los baselines simples en F1 y ROC-AUC, por lo que su complejidad se justifica frente a reglas triviales.
- La brecha de F1 entre entrenamiento y validación se reduce conforme aumenta el tamaño muestral, lo que sugiere un sobreajuste leve al inicio y un ajuste razonable con más datos.
- Persisten brechas entre grupos, especialmente por país y categoría. Estas brechas deben interpretarse junto con diferencias de prevalencia real y cobertura muestral, no como prueba automática de discriminación.
- Las mejoras recomendadas son: calibración por grupo, validación temporal/geográfica, reponderación de grupos subrepresentados, revisión humana cuando el modelo opere fuera de distribución y monitoreo periódico de fairness.